In [3]:
import pandas as pd
import numpy as np 
import os 
import requests
from  bs4 import BeautifulSoup

In [4]:
rows = []

url = f'https://en.wikipedia.org/wiki/List_of_animated_feature_films_of_2022'

# User agent qurish / Building user agent 
headers = {
    'User-Agent': 'Mozilla/5.0'
}

response = requests.get(url=url, headers=headers)
response

<Response [200]>

In [8]:
soup = BeautifulSoup(response.text, 'html.parser')
table = soup.find('table', {"class": "wikitable sortable jquery-tablesorter"})
table

In [10]:
headers =  [th.get_text(strip = True) for th in table.find_all('th')]

for row in table.find('tbody').find_all('tr'):
    cols = [td.get_text(strip = True) for td in row.find_all('td')]
    if cols:
        rows.append(cols)

df = pd.DataFrame(rows, columns=headers[:len(rows[0])])

AttributeError: 'NoneType' object has no attribute 'find_all'

In [ ]:
import time
import requests
import pandas as pd
from bs4 import BeautifulSoup

YEARS = [2010,2011, 2012, 2013, 2014, 2015,2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]

BASE_URL = "https://en.wikipedia.org/wiki/List_of_animated_feature_films_of_{year}"

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/120.0.0.0 Safari/537.36"
    )
}

DELAY_BETWEEN_REQUESTS = 1.5


#Table parser
def parse_table_with_spans(table) -> pd.DataFrame:

    grid: dict[tuple[int, int], str] = {}
    max_col = 0

    for row_idx, tr in enumerate(table.find_all("tr")):
        col_idx = 0
        for cell in tr.find_all(["th", "td"]):
            # Skip positions already filled by a prior rowspan
            while grid.get((row_idx, col_idx)) is not None:
                col_idx += 1

            text     = cell.get_text(separator=" ", strip=True)
            rowspan  = int(cell.get("rowspan", 1))
            colspan  = int(cell.get("colspan", 1))

            for r in range(rowspan):
                for c in range(colspan):
                    grid[(row_idx + r, col_idx + c)] = text

            col_idx += colspan
            max_col  = max(max_col, col_idx)

    if not grid:
        return pd.DataFrame()

    max_row    = max(r for r, _ in grid) + 1
    table_data = [
        [grid.get((r, c), "") for c in range(max_col)]
        for r in range(max_row)
    ]

    headers = table_data[0]
    rows    = table_data[1:]

    # Deduplicate column names
    seen: dict[str, int] = {}
    clean_headers = []
    for h in headers:
        if h in seen:
            seen[h] += 1
            clean_headers.append(f"{h}.{seen[h]}")
        else:
            seen[h] = 0
            clean_headers.append(h)

    return pd.DataFrame(rows, columns=clean_headers)


# ── Single-year scraper ────────────────────────────────────────────────────────
def scrape_year(year: int) -> pd.DataFrame:
    """
    Fetches the Wikipedia page for *year* and extracts the wikitable
    directly after the <h2 id="List"> heading.
    Adds a 'Year' column for easy identification in the combined file.
    """
    url = BASE_URL.format(year=year)
    print(f"  Fetching {url} ...", end=" ", flush=True)

    response = requests.get(url, headers=HEADERS, timeout=15)
    response.raise_for_status()

    soup = BeautifulSoup(response.text, "html.parser")

    # Locate the "List" heading (modern Wikipedia wraps it in a div)
    list_heading = soup.find("h2", id="List")
    if list_heading is None:
        raise ValueError(f"[{year}] Could not find the 'List' section heading.")

    search_start = list_heading.find_parent("div", class_="mw-heading") or list_heading

    # Find the first wikitable after that heading
    table = None
    for sibling in search_start.find_next_siblings():
        if sibling.name == "table" and "wikitable" in sibling.get("class", []):
            table = sibling
            break
        if sibling.name == "h2":   # reached the next major section — stop
            break

    if table is None:
        raise ValueError(f"[{year}] Could not find a wikitable after the 'List' heading.")

    df = parse_table_with_spans(table)
    df.insert(0, "Year", year)          # prepend Year column
    print(f"✅ {len(df)} rows, {len(df.columns)} columns")
    return df


# ── Main ───────────────────────────────────────────────────────────────────────
def main():
    all_frames: list[pd.DataFrame] = []

    print(f"Scraping animated feature film lists for years: {YEARS}\n")

    for i, year in enumerate(YEARS):
        try:
            df = scrape_year(year)

            # Save individual CSV
            filename = f"animated_films_{year}.csv"
            df.to_csv(filename, index=False, encoding="utf-8-sig")
            print(f"     💾 Saved → {filename}")

            all_frames.append(df)

        except Exception as exc:
            print(f"  ❌ Error scraping {year}: {exc}")

        # Polite delay between requests (skip after last year)
        if i < len(YEARS) - 1:
            time.sleep(DELAY_BETWEEN_REQUESTS)

    # Save combined CSV
    if all_frames:
        combined = pd.concat(all_frames, ignore_index=True)
        combined_file = "animated_films_all.csv"
        combined.to_csv(combined_file, index=False, encoding="utf-8-sig")
        print(f"\n📦 Combined file saved → {combined_file}")
        print(f"   Total rows: {len(combined)} across {len(all_frames)} year(s)")

        print("\nColumn names:")
        for col in combined.columns:
            print(f"  - {col}")

    return combined if all_frames else pd.DataFrame()


if __name__ == "__main__":
    main()

Scraping animated feature film lists for years: [2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]

  Fetching https://en.wikipedia.org/wiki/List_of_animated_feature_films_of_2010 ... ✅ 112 rows, 10 columns
     💾 Saved → animated_films_2010.csv
  Fetching https://en.wikipedia.org/wiki/List_of_animated_feature_films_of_2011 ... ✅ 131 rows, 10 columns
     💾 Saved → animated_films_2011.csv
  Fetching https://en.wikipedia.org/wiki/List_of_animated_feature_films_of_2012 ... ✅ 129 rows, 10 columns
     💾 Saved → animated_films_2012.csv
  Fetching https://en.wikipedia.org/wiki/List_of_animated_feature_films_of_2013 ... ✅ 161 rows, 10 columns
     💾 Saved → animated_films_2013.csv
  Fetching https://en.wikipedia.org/wiki/List_of_animated_feature_films_of_2014 ... ✅ 162 rows, 10 columns
     💾 Saved → animated_films_2014.csv
  Fetching https://en.wikipedia.org/wiki/List_of_animated_feature_films_of_2015 ... ✅ 173 rows, 10 columns
     💾 Saved → an

In [65]:
df = pd.read_csv(r'C:\animation_technique_classification\Scraping\animated_films_all.csv')
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2370 entries, 0 to 2369
Data columns (total 10 columns):
 #   Column               Non-Null Count  Dtype
---  ------               --------------  -----
 0   Year                 2370 non-null   int64
 1   Title                2370 non-null   str  
 2   Country              2369 non-null   str  
 3   Director             2324 non-null   str  
 4   Studio               2236 non-null   str  
 5   Animation technique  2352 non-null   str  
 6   Type                 192 non-null    str  
 7   Notes                1725 non-null   str  
 8   Release date         2354 non-null   str  
 9   Duration             2319 non-null   str  
dtypes: int64(1), str(9)
memory usage: 185.3 KB


In [66]:
df.sample(10)

,Year,Title,Country,Director,Studio,Animation technique,Type,Notes,Release date,Duration
1768,2022,3 Little Kungpoo Goats,China Islamic Republic of Iran,Farzad Dalvand Kianoush Dalvand,Aria Animation Studio,CG animation,NaN,[ 1 ],"September 10, 2022 (World Festival of Animated...",89 minutes
1945,2023,Detective Conan: Black Iron Submarine,Japan,Yuzuru Tachikawa,TMS Entertainment,Traditional,NaN,[ 81 ],"April 14, 2023",109 minutes
1157,2017,Smurfs: The Lost Village,United States,Kelly Asbury,Sony Pictures Animation,CGI animation,NaN,[ 182 ],"April 5, 2017 ( Belgium and Asia ) April 7, 20...",90 minutes [ 183 ]
1290,2018,North of Blue [ citation needed ],United States,Joanna Priestley,Joanna Priestley Motion Pictures,Traditional,NaN,[ 161 ],"June 11, 2018 ( Annecy ) [ 162 ]",60 minutes
1040,2017,Barbie: Video Game Hero,United States,Conrad Helten & Zeke Norton,Universal Studios Arc Productions Rainmaker St...,CGI animation,NaN,NaN,"January 31, 2017 (DVD) March 26, 2017 ( Nickel...",72 minutes
1396,2019,"Jacob, Mimmi and the Talking Dogs Jekabs, Mimm...",Latvia Poland,Edmunds Jansons,Atom Art Letko,Traditional,NaN,[ 85 ],"July 20, 2019 ( Giffoni Film Festival ) Septem...",70 minutes [ 86 ]
403,2013,Ctyrlístek ve sluzbách krále [ cs ],Czech Republic,Michal Zabka [ cs ],NaN,CG animation,NaN,NaN,"February 28, 2013 (Czech) October 16, 2013 (DV...",90 minutes
1759,2021,Tropical-Rouge! Pretty Cure the Movie: The Sno...,Japan,Junji Shimizu,Toei Animation,Traditional,NaN,[ 295 ] [ 296 ] [ 297 ],"October 23, 2021",70 minutes [ 298 ]
1357,2019,BoBoiBoy Movie 2,Malaysia,Nizam Razak,Animonsta Studios,CG animation,NaN,[ 27 ],"August 8, 2019",110 minutes
1206,2018,Captain Morten and the Spider Queen Kapten Mor...,Estonia Ireland United Kingdom Belgium,Kaspar Jancis,"Nukufilm, Telegael, Calon, Grid Animation",Stop Motion,NaN,[ 30 ],"June 5, 2018 ( Animafest Zagreb ) March 21, 20...",79 minutes
